
# Seção 2 — Baseline com LLM

Esta seção executa o **baseline LLM-only** do TCC.

## Objetivo científico

Medir o desempenho de um LLM quando ele recebe um evento textual do projeto e precisa identificar, **sem usar o grafo de relações nem as regras da arquitetura híbrida**:

- tipo do evento;
- entidade principal;
- entidades relacionadas;
- entidades potencialmente impactadas;
- artefatos gerenciais potencialmente impactados.

Depois, esse resultado será comparado com a arquitetura proposta:

**LLM + grafo + regras + rastreabilidade**.

> **Regra metodológica importante:** o `ground truth` nunca deve ser enviado ao LLM.  
> Ele é usado somente depois da inferência, para avaliação.


## 15. Preparar o ambiente local

Use as dependências de experiments do repositório, instaladas antes da execução.


In [ ]:
from pathlib import Path
import sys

# Descobre a raiz a partir da pasta local, sem caminhos pessoais fixos.
REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "TCC_Codigos/data/project_graph.json").is_file()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Abra o projeto a partir da raiz do repositório TCC.")
PROJECT_DIR = REPO_ROOT / "TCC_Codigos"
if str(PROJECT_DIR / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR / "src"))
print("Projeto local:", PROJECT_DIR)

from datetime import datetime, timezone
from pprint import pprint
import hashlib
import json
import os
import re
import time

import pandas as pd


In [ ]:
print("Ambiente local preparado:", PROJECT_DIR)


## 16. Configurar a chave local

Use OPENAI_API_KEY no .env da raiz ou no ambiente. O valor não é exibido.


In [ ]:
from openai import OpenAI
from tcc_agent.local_runtime import load_local_env

load_local_env(REPO_ROOT)
api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("Configure OPENAI_API_KEY no .env da raiz ou no ambiente local.")
client = OpenAI(api_key=api_key)
print("Cliente configurado a partir do ambiente local. Chave não exibida.")



## 17. Carregar os dados experimentais

Serão carregados:

- os **36 cenários**;
- o catálogo de entidades do Projeto Alfa.

O catálogo de entidades pode ser mostrado ao LLM para que ele devolva IDs válidos, mas **as relações do grafo não serão mostradas ao baseline**.


In [ ]:

scenarios_file = PROJECT_DIR / "data" / "scenarios_ground_truth.json"
graph_file = PROJECT_DIR / "data" / "project_graph.json"

with open(scenarios_file, "r", encoding="utf-8") as f:
    scenarios = json.load(f)

with open(graph_file, "r", encoding="utf-8") as f:
    project_graph = json.load(f)

entity_catalog = [
    {
        "id": node["id"],
        "name": node["name"],
        "type": node["type"],
    }
    for node in project_graph["nodes"]
]

print("Cenários carregados:", len(scenarios))
print("Entidades disponíveis:", len(entity_catalog))

# Conferência simples
print("\nPrimeiro cenário (texto apenas):")
print(scenarios[0]["scenario_id"], "-", scenarios[0]["source_text"])



## 18. Fixar o modelo do baseline

Para a fase de desenvolvimento, deixamos `gpt-6-luna` como padrão por ser um modelo de menor custo.

### Regra experimental

Vocês podem trocar o modelo **durante o piloto**.  
Depois que o prompt e o modelo forem congelados, **não alterem nenhum dos dois durante a execução dos 36 casos**.


In [ ]:

MODEL = "gpt-6-luna"

ALLOWED_EVENT_TYPES = [
    "delay",
    "decision",
    "test_problem",
    "technical_change",
    "action_commitment",
    "risk_signal",
    "supplier_change",
    "informational",
]

ALLOWED_ARTIFACTS = [
    "action_log",
    "risk_register",
    "decision_log",
    "status_report",
]

print("Modelo configurado:", MODEL)



## 19. Definir o prompt v2 do baseline

Este é o contrato experimental.

O baseline recebe:

1. o texto do evento;
2. o catálogo de entidades existentes.

Ele **não recebe**:

- as relações do grafo;
- os caminhos de propagação;
- as regras documentais;
- o ground truth.

A resposta deve ser um único objeto JSON.


In [ ]:

SYSTEM_PROMPT_V2 = """
Você é um analisador de eventos de gestão de projetos.

Sua tarefa é analisar UM evento textual de um projeto de desenvolvimento de produto
e produzir uma previsão estruturada dos possíveis impactos gerenciais.

IMPORTANTE:
- Use apenas as informações fornecidas no evento e no catálogo de entidades.
- Você NÃO possui acesso às relações do grafo do projeto.
- Não invente fatos específicos que não estejam sustentados pelo texto.
- Não assuma automaticamente que todo evento gera impacto.
- Quando o texto apenas comunicar um fato sem consequência gerencial relevante,
  use o tipo "informational".
- Utilize somente IDs presentes no catálogo quando indicar entidades.
- Utilize somente os tipos de evento e artefatos permitidos.
- Retorne APENAS JSON válido.
- Não use Markdown.
- Não inclua comentários fora do JSON.

Tipos de evento permitidos:
delay
decision
test_problem
technical_change
action_commitment
risk_signal
supplier_change
informational

Artefatos permitidos:
action_log
risk_register
decision_log
status_report

DEFINIÇÃO DOS ARTEFATOS:

risk_register:
Use quando o evento introduzir, aumentar, materializar ou alterar um risco relevante
para prazo, custo, qualidade, fornecimento, teste, entrega ou desempenho do projeto.

status_report:
Use quando o evento alterar ou puder alterar o estado atual de uma parte relevante
do projeto, como prazo, fornecedor, componente, teste, milestone, entrega ou qualidade.

decision_log:
Use quando o texto indicar que uma decisão foi tomada, aprovada, rejeitada,
substituída ou formalmente alterada.

action_log:
Use quando houver uma ação explícita a executar, um compromisso assumido,
um responsável definido ou uma ação corretiva que precise ser acompanhada.

REGRAS IMPORTANTES PARA ARTEFATOS:
- Não selecione action_log apenas porque seria razoável tomar alguma ação.
- Não selecione decision_log se nenhuma decisão foi tomada.
- Não selecione risk_register se o texto não indicar risco, impacto potencial,
  problema relevante ou incerteza gerencial.
- Não selecione status_report se o evento não alterar ou potencialmente alterar
  o estado do projeto.
- É permitido retornar impacted_artifacts vazio.
- Se o texto deixar explícito que a situação não afeta prazo, risco, decisão,
  ação ou status do projeto, retorne impacted_artifacts vazio quando apropriado.

DEFINIÇÃO DE "informational":
Use "informational" quando a informação apenas comunicar um fato que não altera
o estado gerencial do projeto e não exige atualização de risco, decisão, ação
ou status.

Não transforme automaticamente toda menção a fornecedor, componente, prazo,
atividade ou mudança em um evento de impacto.

Formato obrigatório:
{
  "event_type": "string",
  "primary_entity": "ID ou null",
  "related_entities": ["ID"],
  "impacted_entities": ["ID"],
  "impacted_artifacts": ["artifact"],
  "responsible": "ID ou null",
  "due_date": "YYYY-MM-DD ou null",
  "effective_date": "YYYY-MM-DD ou null",
  "rationale": "justificativa curta"
}

REGRAS PARA ENTIDADES:
- primary_entity deve representar a entidade principal diretamente envolvida no evento.
- related_entities deve conter apenas entidades claramente relacionadas ao texto.
- impacted_entities deve conter apenas entidades que você acredita poderem ser afetadas
  com base no texto fornecido.
- Não invente relações de dependência que não estejam descritas no texto.

REGRAS PARA DATAS E RESPONSÁVEIS:
- responsible deve ser null se nenhum responsável estiver explicitamente indicado.
- due_date deve ser null se não houver prazo explícito.
- effective_date deve ser null se não houver data efetiva explícita.

REGRAS FINAIS:
- impacted_artifacts deve conter apenas artefatos que realmente precisam ser revisados.
- impacted_entities pode ser vazio.
- impacted_artifacts pode ser vazio.
- Não inclua o ground truth, porque ele não é fornecido a você.
""".strip()

prompt_hash = hashlib.sha256(
    SYSTEM_PROMPT_V2.encode("utf-8")
).hexdigest()[:16]

print("Prompt:", "BASELINE_V2")
print("SHA-256 (16 chars):", prompt_hash)
print("\n--- PROMPT ---\n")
print(SYSTEM_PROMPT_V2)



## 20. Criar funções auxiliares

Estas funções:

- montam a entrada do modelo;
- extraem JSON da resposta;
- validam o formato básico;
- executam um cenário;
- registram metadados do experimento.


In [ ]:

def build_user_input(scenario):
    # IMPORTANTE: usamos apenas o texto/fonte/data do evento.
    # Campos expected_* nunca entram no prompt.
    catalog_text = "\n".join(
        f"- {e['id']} | {e['type']} | {e['name']}"
        for e in entity_catalog
    )

    return f"""
CENÁRIO
ID: {scenario['scenario_id']}
Data: {scenario.get('occurred_at')}
Fonte: {scenario.get('source_type')}
Texto: {scenario['source_text']}

CATÁLOGO DE ENTIDADES VÁLIDAS
{catalog_text}

Analise o cenário e retorne somente o JSON solicitado.
""".strip()


def extract_json(text):
    text = text.strip()

    # Remove fences caso o modelo os use apesar da instrução.
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)

    return json.loads(text)


def validate_prediction(pred):
    required = {
        "event_type",
        "primary_entity",
        "related_entities",
        "impacted_entities",
        "impacted_artifacts",
        "responsible",
        "due_date",
        "effective_date",
        "rationale",
    }

    missing = required - set(pred)
    if missing:
        raise ValueError(f"Campos ausentes: {sorted(missing)}")

    if pred["event_type"] not in ALLOWED_EVENT_TYPES:
        raise ValueError(f"event_type inválido: {pred['event_type']}")

    invalid_artifacts = set(pred["impacted_artifacts"]) - set(ALLOWED_ARTIFACTS)
    if invalid_artifacts:
        raise ValueError(f"Artefatos inválidos: {sorted(invalid_artifacts)}")

    valid_ids = {e["id"] for e in entity_catalog}

    ids_to_check = []
    if pred["primary_entity"] is not None:
        ids_to_check.append(pred["primary_entity"])
    ids_to_check.extend(pred["related_entities"])
    ids_to_check.extend(pred["impacted_entities"])
    if pred["responsible"] is not None:
        ids_to_check.append(pred["responsible"])

    invalid_ids = set(ids_to_check) - valid_ids
    if invalid_ids:
        raise ValueError(f"IDs não existentes no catálogo: {sorted(invalid_ids)}")

    return True


def run_scenario(scenario):
    user_input = build_user_input(scenario)

    started = time.time()

    response = client.responses.create(
        model=MODEL,
        input=[
            {"role": "system", "content": SYSTEM_PROMPT_V2},
            {"role": "user", "content": user_input},
        ],
    )

    elapsed = time.time() - started
    raw_text = response.output_text

    record = {
        "scenario_id": scenario["scenario_id"],
        "model": MODEL,
        "prompt_version": "BASELINE_V2",
        "prompt_hash": prompt_hash,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "elapsed_seconds": round(elapsed, 3),
        "response_id": getattr(response, "id", None),
        "raw_output": raw_text,
        "parse_ok": False,
        "validation_ok": False,
        "prediction": None,
        "error": None,
    }

    try:
        pred = extract_json(raw_text)
        record["parse_ok"] = True
        validate_prediction(pred)
        record["validation_ok"] = True
        record["prediction"] = pred
    except Exception as exc:
        record["error"] = str(exc)

    return record



## 21. Testar UM cenário antes do piloto

Primeiro vamos testar somente `S001`.

Isso serve para confirmar:

- conexão com a API;
- formato da resposta;
- parsing do JSON;
- validação dos IDs.

**Não use o ground truth para corrigir a resposta do modelo nesta etapa.**


In [ ]:
# Diagnósticos de rede não são executados automaticamente.
print("Configuração local carregada; o próximo teste chama somente o experimento.")


In [ ]:
print("Cliente local já configurado; chave não exibida.")


In [ ]:
scenario_001 = next(
    s for s in scenarios
    if s["scenario_id"] == "S001"
)

print("Texto enviado:")
print(scenario_001["source_text"])

try:
    test_result = run_scenario(scenario_001)

    print("\nResultado técnico:")
    pprint(test_result)

except Exception as e:
    print("\nOcorreu um erro durante a chamada do modelo:")
    print(type(e).__name__)
    print(e)


## 22. Executar os 6 casos piloto

Casos escolhidos:

- S001
- S003
- S007
- S009
- S024
- S035

Eles incluem situações fáceis, difíceis e casos em que o sistema deve evitar gerar atualizações desnecessárias.

### Atenção

O piloto existe para vocês avaliarem **se o prompt é compreensível e se o JSON é estável**.

Após eventuais ajustes, criem uma nova versão do prompt, registrem as mudanças e então **congelem** a versão final.


In [ ]:

PILOT_IDS = ["S001", "S003", "S007", "S009", "S024", "S035"]

pilot_records = []

for sid in PILOT_IDS:
    scenario = next(s for s in scenarios if s["scenario_id"] == sid)

    print(f"Executando {sid}...")
    record = run_scenario(scenario)
    pilot_records.append(record)

    print(
        "  parse_ok =", record["parse_ok"],
        "| validation_ok =", record["validation_ok"]
    )

print("\nPiloto concluído.")


In [ ]:

# Visualização resumida do piloto

pilot_table = []

for r in pilot_records:
    pred = r.get("prediction") or {}

    pilot_table.append({
        "scenario_id": r["scenario_id"],
        "parse_ok": r["parse_ok"],
        "validation_ok": r["validation_ok"],
        "event_type": pred.get("event_type"),
        "primary_entity": pred.get("primary_entity"),
        "impacted_artifacts": ", ".join(pred.get("impacted_artifacts", [])),
        "error": r.get("error"),
    })

pd.DataFrame(pilot_table)



## 23. Salvar o piloto

As respostas precisam ser registradas antes de qualquer análise posterior.

Isso cria um **audit trail** do experimento.


In [ ]:

OUTPUT_DIR = PROJECT_DIR / "outputs" / "baseline_v2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Salva o prompt congelável
(OUTPUT_DIR / "system_prompt_v2.txt").write_text(
    SYSTEM_PROMPT_V2,
    encoding="utf-8"
)

# Metadados
metadata = {
    "model": MODEL,
    "prompt_version": "BASELINE_V2",
    "prompt_hash": prompt_hash,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "pilot_ids": PILOT_IDS,
}

with open(OUTPUT_DIR / "metadata_v2.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

# Piloto completo em JSONL
with open(OUTPUT_DIR / "pilot_results_v2.jsonl", "w", encoding="utf-8") as f:
    for record in pilot_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("Arquivos salvos em:")
print(OUTPUT_DIR)



## 24. CONGELAMENTO DO EXPERIMENTO

Antes de executar os 36 casos, vocês dois devem confirmar:

- [ ] o ground truth foi revisado pelos dois autores;
- [ ] divergências foram reconciliadas;
- [ ] o ground truth foi congelado;
- [ ] o modelo está definido;
- [ ] o prompt final está definido;
- [ ] o formato JSON está estável no piloto;
- [ ] nenhum campo `expected_*` é enviado ao LLM.

### Só depois disso execute a próxima célula.

Para evitar execução acidental, `RUN_FULL_EXPERIMENT` começa como `False`.


In [ ]:

RUN_FULL_EXPERIMENT = False

print("RUN_FULL_EXPERIMENT =", RUN_FULL_EXPERIMENT)

if not RUN_FULL_EXPERIMENT:
    print(
        "\nExperimento completo BLOQUEADO. "
        "Mude para True somente depois do congelamento."
    )



## 25. Executar os 36 cenários

Quando estiverem prontos:

```python
RUN_FULL_EXPERIMENT = True
```

e executem a célula abaixo.

Ela salva cada resultado imediatamente, reduzindo o risco de perder dados se a sessão do ambiente local cair.


In [ ]:

full_results_file = OUTPUT_DIR / "full_results_v2.jsonl"

if RUN_FULL_EXPERIMENT:
    full_records = []

    # Reinicia o arquivo de resultados desta execução.
    full_results_file.write_text("", encoding="utf-8")

    for idx, scenario in enumerate(scenarios, start=1):
        sid = scenario["scenario_id"]
        print(f"[{idx:02d}/{len(scenarios)}] Executando {sid}...")

        try:
            record = run_scenario(scenario)
        except Exception as exc:
            record = {
                "scenario_id": sid,
                "model": MODEL,
                "prompt_version": "BASELINE_V2",
                "prompt_hash": prompt_hash,
                "timestamp_utc": datetime.now(timezone.utc).isoformat(),
                "parse_ok": False,
                "validation_ok": False,
                "prediction": None,
                "raw_output": None,
                "error": f"API_ERROR: {exc}",
            }

        full_records.append(record)

        # Persistência incremental.
        with open(full_results_file, "a", encoding="utf-8") as f:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

        # Pausa curta para execução conservadora.
        time.sleep(0.25)

    print("\nExecução completa finalizada.")
    print("Resultados:", full_results_file)

else:
    print("Nada foi executado porque RUN_FULL_EXPERIMENT = False.")



## 26. Carregar os resultados salvos

Esta célula funciona mesmo se o ambiente local tiver sido reiniciado, desde que o arquivo de resultados permaneça disponível no checkout local.


In [ ]:

if full_results_file.exists() and full_results_file.stat().st_size > 0:
    with open(full_results_file, "r", encoding="utf-8") as f:
        full_records = [json.loads(line) for line in f if line.strip()]

    print("Resultados carregados:", len(full_records))
else:
    full_records = []
    print("Ainda não existem resultados completos salvos.")



## 27. Calcular as métricas do baseline

### Métricas principais

1. **Acurácia do tipo de evento**
2. **Precision de artefatos**
3. **Recall de artefatos**
4. **F1 de artefatos**

### Interpretação simples

**Precision**  
Entre os artefatos que a IA disse que estavam impactados, quantos realmente estavam?

**Recall**  
Entre os artefatos que realmente deveriam ser impactados, quantos a IA encontrou?

**F1**  
Equilibra precision e recall.


In [ ]:

def safe_set(values):
    return set(values or [])


def evaluate_baseline(records, scenarios):
    gt_by_id = {s["scenario_id"]: s for s in scenarios}

    rows = []

    event_correct = 0
    event_total = 0

    artifact_tp = 0
    artifact_fp = 0
    artifact_fn = 0

    for record in records:
        sid = record["scenario_id"]
        gt = gt_by_id[sid]
        pred = record.get("prediction")

        if not pred:
            # Resposta inválida: conta como erro e perde todos os artefatos esperados.
            expected_artifacts = safe_set(gt["expected_artifacts"])
            artifact_fn += len(expected_artifacts)

            rows.append({
                "scenario_id": sid,
                "parse_ok": record.get("parse_ok"),
                "validation_ok": record.get("validation_ok"),
                "event_correct": False,
                "artifact_precision": 0.0,
                "artifact_recall": 0.0 if expected_artifacts else 1.0,
                "artifact_f1": 0.0,
                "error": record.get("error"),
            })
            event_total += 1
            continue

        # Tipo do evento
        event_total += 1
        event_is_correct = pred["event_type"] == gt["expected_event_type"]

        if event_is_correct:
            event_correct += 1

        # Artefatos
        expected = safe_set(gt["expected_artifacts"])
        predicted = safe_set(pred["impacted_artifacts"])

        tp = len(expected & predicted)
        fp = len(predicted - expected)
        fn = len(expected - predicted)

        artifact_tp += tp
        artifact_fp += fp
        artifact_fn += fn

        p = tp / (tp + fp) if (tp + fp) else (1.0 if not expected else 0.0)
        r = tp / (tp + fn) if (tp + fn) else 1.0
        f1 = 2 * p * r / (p + r) if (p + r) else 0.0

        rows.append({
            "scenario_id": sid,
            "parse_ok": record.get("parse_ok"),
            "validation_ok": record.get("validation_ok"),
            "event_correct": event_is_correct,
            "artifact_precision": round(p, 4),
            "artifact_recall": round(r, 4),
            "artifact_f1": round(f1, 4),
            "expected_artifacts": sorted(expected),
            "predicted_artifacts": sorted(predicted),
            "error": record.get("error"),
        })

    event_accuracy = event_correct / event_total if event_total else 0.0

    micro_precision = (
        artifact_tp / (artifact_tp + artifact_fp)
        if (artifact_tp + artifact_fp) else 0.0
    )

    micro_recall = (
        artifact_tp / (artifact_tp + artifact_fn)
        if (artifact_tp + artifact_fn) else 0.0
    )

    micro_f1 = (
        2 * micro_precision * micro_recall / (micro_precision + micro_recall)
        if (micro_precision + micro_recall) else 0.0
    )

    summary = {
        "n_scenarios": event_total,
        "event_type_accuracy": event_accuracy,
        "artifact_precision_micro": micro_precision,
        "artifact_recall_micro": micro_recall,
        "artifact_f1_micro": micro_f1,
        "artifact_tp": artifact_tp,
        "artifact_fp": artifact_fp,
        "artifact_fn": artifact_fn,
    }

    return summary, pd.DataFrame(rows)


if full_records:
    summary, detailed_df = evaluate_baseline(full_records, scenarios)

    print("RESULTADOS DO BASELINE\n")
    for key, value in summary.items():
        if isinstance(value, float):
            print(f"{key}: {value:.4f}")
        else:
            print(f"{key}: {value}")

    display(detailed_df)

else:
    print("Execute os 36 cenários antes de calcular as métricas.")



## 28. Salvar métricas e tabela detalhada

Os arquivos produzidos aqui poderão ser usados diretamente na análise do TCC.


In [ ]:

if full_records:
    metrics_file = OUTPUT_DIR / "baseline_metrics_v2.json"
    details_file = OUTPUT_DIR / "baseline_details_v2.csv"

    with open(metrics_file, "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    detailed_df.to_csv(details_file, index=False, encoding="utf-8-sig")

    print("Métricas:", metrics_file)
    print("Detalhes:", details_file)
else:
    print("Nenhuma métrica foi salva porque o experimento completo ainda não foi executado.")



## 29. Análise de erros

Não basta olhar apenas o F1.

Depois da execução, revisem especialmente:

- falsos positivos;
- falsos negativos;
- eventos `informational`;
- eventos difíceis;
- IDs de entidades incorretos;
- documentos desnecessariamente marcados;
- impactos que o LLM não conseguiu inferir sem o grafo.

Essa análise ajudará a explicar **por que a arquitetura híbrida pode ser necessária**.


In [ ]:

if full_records:
    error_view = detailed_df[
        (detailed_df["event_correct"] == False)
        | (detailed_df["artifact_f1"] < 1.0)
        | (detailed_df["validation_ok"] == False)
    ].copy()

    print("Cenários com algum erro/diferença:", len(error_view))
    display(error_view)
else:
    print("Ainda não há resultados completos para analisar.")



# Checklist de encerramento da Seção 2

Antes de iniciar a arquitetura híbrida, confirmem:

- [ ] Chave da API armazenada no .env local ou ambiente.
- [ ] Modelo registrado.
- [ ] Prompt final congelado.
- [ ] Hash do prompt salvo.
- [ ] Ground truth congelado antes dos 36 casos.
- [ ] 36 respostas salvas.
- [ ] JSON validado.
- [ ] Acurácia do tipo de evento calculada.
- [ ] Precision, Recall e F1 de artefatos calculados.
- [ ] Falsos positivos e falsos negativos revisados.
- [ ] Resultados exportados para outputs/baseline_v2 no checkout local.

## Próxima seção

**Seção 3 — Arquitetura híbrida**

O mesmo conjunto de 36 cenários será executado com:

**LLM → evento estruturado → grafo → propagação → regras documentais → rastreabilidade**

Depois, vocês compararão o resultado da Seção 3 com este baseline.
